# NAQSHKASH Stage 1 — Colab
Choose **Runtime → Change runtime type → T4 GPU**, then run cells in order. Configure links below. Full training commands are displayed only, even with Run All. No upload is performed automatically.

Free Colab can disconnect unpredictably. A browser disconnect differs from runtime termination; keeping a browser open does not guarantee runtime survival. Mid-epoch saves limit lost progress only when copies survive outside `/content`. Use separate persistent folders for Kaggle and Colab runs; never let both overwrite the same checkpoint pair.

In [ ]:
import os, sys, subprocess
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available(), 'CUDA version:', torch.version.cuda)
assert torch.cuda.is_available(), 'Enable a GPU runtime before continuing'
print('GPU:', torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info(0)
print('GPU GiB free/total:', free / 2**30, total / 2**30)

In [ ]:
PROJECT_ROOT = Path('/content/NAQSHKASH_PROJECT')
DATASET_ROOT = Path('/content/NAQSHKASH_DATA')
OUTPUT_ROOT = Path('/content/naqshkash_colab')
PERSISTENT_CKPT_ROOT = None  # e.g. Path('/content/drive/MyDrive/NAQSHKASH/colab_run_01')
PROJECT_ZIP_URL = ''
PROJECT_FILE_ID = ''
DATASET_ZIP_URL = ''
DATASET_FILE_ID = ''
MOUNT_DRIVE = False
RESTORE_CHECKPOINTS = False  # fresh runtime: restore full_run before resume
UPLOAD_BACKUP = False       # manual fallback: upload checkpoints.zip saved on your computer
DOWNLOAD_BACKUP = False     # set True and rerun backup cell while runtime is alive
WIDTH_BUDGET = 16384
MAX_BATCH_SIZE = 32
WORKERS = 2
# Colab already supplies CUDA torch. Install only runtime dependencies; do not pip install -r requirements.txt.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown', 'numpy', 'opencv-python-headless'], check=True)

Optional Drive mount. If credentials fail, keep the runtime and use manual ZIP download/upload cells. A gdown share link downloads files; it does not provide writable checkpoint storage. Manual backups require your action before the runtime disappears.

In [ ]:
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as error:
        print('Drive mount failed:', error)
        PERSISTENT_CKPT_ROOT = None
if PERSISTENT_CKPT_ROOT is not None:
    PERSISTENT_CKPT_ROOT = Path(PERSISTENT_CKPT_ROOT)
    assert Path('/content/drive/MyDrive').is_dir(), 'Mount Drive first, or configure another persistent filesystem explicitly'
    PERSISTENT_CKPT_ROOT.mkdir(parents=True, exist_ok=True)
    probe = PERSISTENT_CKPT_ROOT / '.write_probe'
    probe.write_text('ok'); probe.unlink()
    print('Automatic persistence:', PERSISTENT_CKPT_ROOT)
else:
    print('MANUAL BACKUP MODE: /content will be lost. Download a ZIP after controlled runs and periodically during training.')

In [ ]:
import zipfile
import gdown
def acquire(destination, url, file_id):
    destination.mkdir(parents=True, exist_ok=True)
    if not (url or file_id):
        print('Using existing extracted files:', destination)
        return
    archive = destination.parent / (destination.name + '.zip')
    result = (gdown.download(id=file_id, output=str(archive), quiet=False) if file_id
              else gdown.download(url=url, output=str(archive), fuzzy=True, quiet=False))
    if result is None:
        raise RuntimeError('gdown failed: check sharing permissions, file ID and download quota')
    with zipfile.ZipFile(archive) as source:
        base = destination.resolve()
        for member in source.infolist():
            if not (base / member.filename).resolve().is_relative_to(base):
                raise ValueError('Unsafe ZIP member: ' + member.filename)
        source.extractall(destination)
def detect(base, required):
    candidates = [base] + [p.parent for p in base.rglob(required[0]) if p.is_dir()]
    matches = sorted(set(p for p in candidates if all((p / item).exists() for item in required)))
    if len(matches) != 1:
        raise ValueError(f'Expected one root containing {required}; found {matches}. Check extraction/path.')
    return matches[0]
acquire(PROJECT_ROOT, PROJECT_ZIP_URL, PROJECT_FILE_ID)
PROJECT_ROOT = detect(PROJECT_ROOT, ['training', 'models', 'data'])
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
acquire(DATASET_ROOT, DATASET_ZIP_URL, DATASET_FILE_ID)
DATASET_ROOT = detect(DATASET_ROOT, ['train', 'validation', 'test'])
for split in ('train', 'validation', 'test'):
    for item in ('images', 'labels', 'metadata.csv'):
        assert (DATASET_ROOT / split / item).exists(), f'Missing {split}/{item}: check DATASET_ROOT'
print('Project:', PROJECT_ROOT, 'Dataset:', DATASET_ROOT)
subprocess.run([sys.executable, '-m', 'training.train_crnn', '--help'], check=True)

In [ ]:
from training.gpu_deployment import display_command
from training.colab_checkpoints import backup, restore
def inspect(directory):
    result = None
    for name in ('last.pt', 'best.pt'):
        path = Path(directory) / name
        if not path.exists():
            print(name, 'absent (best appears after first validation)'); continue
        ckpt = torch.load(path, map_location='cpu', weights_only=True)
        info = {key: ckpt[key] for key in ('epoch', 'next_batch', 'global_step')}
        print(name, info)
        for key in ('optimizer_state_dict', 'scheduler_state_dict', 'scaler_state_dict'):
            print(key, 'present:', key in ckpt, 'nonempty:', bool(ckpt.get(key)))
        assert ckpt.get('optimizer_state_dict', {}).get('state'), 'Missing optimizer state'
        assert ckpt.get('scheduler_state_dict'), 'Missing scheduler state'
        assert ckpt.get('scaler_state_dict'), 'Expected CUDA AMP scaler state'
        if name == 'last.pt': result = info
        del ckpt
    return result
base = [sys.executable, '-m', 'training.train_crnn', '--dataset-root', str(DATASET_ROOT),
        '--width-budget', str(WIDTH_BUDGET), '--max-batch-size', str(MAX_BATCH_SIZE), '--amp', '--num-workers', str(WORKERS)]
def command(run, *extra):
    cmd = base + ['--output-root', str(OUTPUT_ROOT / run)] + list(extra)
    if PERSISTENT_CKPT_ROOT is not None:
        cmd += ['--checkpoint-mirror-dir', str(PERSISTENT_CKPT_ROOT / run)]
    return cmd
smoke = ['--epochs', '30', '--train-limit', '32768', '--val-limit', '256', '--skip-test',
         '--checkpoint-every-batches', '100', '--stop-after-batches', '300']
plan = {
    'preflight': command('quick_probe', '--preflight-only', '--train-limit', '256', '--val-limit', '32'),
    'interrupt': command('resume_test', *smoke),
    'resume_test': command('resume_test', *smoke, '--resume', str(OUTPUT_ROOT / 'resume_test/last.pt')),
    'full_width_probe': command('full_width_probe', '--preflight-only', '--val-limit', '32'),
    'full': command('full_run', '--epochs', '30', '--checkpoint-every-batches', '1000'),
    'resume': command('full_run', '--epochs', '30', '--checkpoint-every-batches', '1000', '--resume', str(OUTPUT_ROOT / 'full_run/last.pt')),
}
def run_short(name):
    assert name in ('preflight', 'interrupt', 'resume_test', 'full_width_probe')
    print(display_command(plan[name]), flush=True)
    subprocess.run(plan[name], cwd=PROJECT_ROOT, check=True)

Quick preflight reads only 256 TRAIN metadata rows/labels/images and 32 validation rows. It tests AMP, CTC backward and Adam allocations on the largest padded batch and longest image in that subset. It creates no checkpoint. It cannot certify unseen width outliers. The optional full-width probe below and every actual run probe full selected TRAIN widths. Budget 16384 is a starting point; reduce it and/or max batch size on OOM.

In [ ]:
run_short('preflight')

Controlled test: 300 batches, then another 300 using the same limits/settings. Use a clean `resume_test` folder for the first run. This is an intentional save-and-exit test, not a hard-kill durability test. Full dataset metadata/vocabulary startup scans can take time here. Smoke checkpoints cannot resume full training because limits differ.

In [ ]:
run_short('interrupt')
before = inspect(OUTPUT_ROOT / 'resume_test')
assert before is not None and before['global_step'] == 300 and before['next_batch'] > 0

In [ ]:
run_short('resume_test')
after = inspect(OUTPUT_ROOT / 'resume_test')
assert after['global_step'] == before['global_step'] + 300
assert after['epoch'] == before['epoch'] and after['next_batch'] == before['next_batch'] + 300

Optional full-width check: reads full TRAIN vocabulary/metadata/widths and probes the worst batches. It can be slow the first time. No training checkpoint is created. The full training command also performs this check automatically.

In [ ]:
RUN_FULL_WIDTH_PROBE = False
if RUN_FULL_WIDTH_PROBE:
    run_short('full_width_probe')

Restore after reset: repeat setup/download cells, then set RESTORE_CHECKPOINTS=True. Restore the atomic checkpoints.zip bundle into a clean full_run folder; it holds a matching last/best pair. With Drive unavailable, set UPLOAD_BACKUP=True to select the ZIP previously downloaded to your computer. Only restore trusted checkpoints.

In [ ]:
if RESTORE_CHECKPOINTS:
    archive = None
    if UPLOAD_BACKUP:
        from google.colab import files
        uploaded = files.upload()
        assert len(uploaded) == 1, 'Select exactly one checkpoints.zip'
        archive = Path(next(iter(uploaded)))
        del uploaded
    elif PERSISTENT_CKPT_ROOT is not None:
        archive = PERSISTENT_CKPT_ROOT / 'full_run/checkpoints.zip'
    assert archive is not None and archive.is_file(), 'No backup found: configure Drive or upload your ZIP'
    restore(archive, OUTPUT_ROOT / 'full_run')
    inspect(OUTPUT_ROOT / 'full_run')

Full training and normal resume — display only. Copy exactly one printed command into a new code cell prefixed with `!` and execute deliberately. Fresh training uses all 534,582 TRAIN samples, full validation, best validation CER, and existing final TEST behavior. The epoch target is 30; existing early stopping remains active. Resume validates settings and uses the saved sampler batch offset without loading skipped training batches; metadata/vocabulary/width identity checks still run.

In [ ]:
for name in ('full', 'resume'):
    print(name + ':\n!' + display_command(plan[name]) + '\n')

Manual backup: set BACKUP_RUN to resume_test or full_run. Run only while training is stopped, so best/last form a consistent pair. Set DOWNLOAD_BACKUP=True and rerun to download; retain that ZIP outside Colab. During an active run with Drive, download the already committed persistent checkpoints.zip instead. Drive mirroring is synchronous after local atomic save; failures warn visibly while training retains local progress. Keep enough space for temporary copies and the ZIP. A failed mirror means new progress is not yet durable.

In [ ]:
BACKUP_RUN = 'resume_test'  # change to full_run after stopping full training
if DOWNLOAD_BACKUP:
    from google.colab import files
    archive = backup(OUTPUT_ROOT / BACKUP_RUN, OUTPUT_ROOT / (BACKUP_RUN + '_checkpoints.zip'))
    files.download(str(archive))
else:
    print('Backup is manual unless persistent mirroring is configured. Set DOWNLOAD_BACKUP=True to download.')